# Time-Series Resampling & Interpolation

This notebook standardises the raw device-telemetry dataset to a **uniform 4-second time grid**.
Raw samples are collected at irregular intervals; downstream clustering and classification
pipelines expect evenly spaced observations.

**Pipeline:**
1. Load `dataset.csv`
2. Parse and sort by `timestamp`
3. Set `timestamp` as the DataFrame index
4. Resample to a 4-second grid → forward-fill → linear interpolate
5. Save the regularised dataset to `dataset_time_series.csv`

**Input:** `../data/dataset/dataset.csv`  
**Output:** `../data/dataset/dataset_time_series.csv`

## Import Library

In [1]:
import pandas as pd  # Only dependency: used for all data loading, resampling, and I/O


## Load Dataset

Load the raw telemetry CSV and preview its structure. Each row represents one sensor
reading with 11 features: `timestamp`, `cpu_usage`, `memory_usage`, `temperature`,
`battery_level`, `battery_drain`, `battery_charging`, `upload_speed_kbps`,
`download_speed_kbps`, `screen_on`, and `brightness`.

In [2]:
# Load the raw dataset; timestamps are initially read as plain strings
df = pd.read_csv('../data/dataset/dataset.csv')
df.head()


,timestamp,cpu_usage,memory_usage,temperature,battery_level,battery_drain,battery_charging,upload_speed_kbps,download_speed_kbps,screen_on,brightness
0,2026-03-09 00:18:05,60,63,36.8,74,-223.0,0,0.000000,0.000000,1,21
1,2026-03-09 00:18:09,49,64,36.8,74,-700.0,0,4.524958,1.739434,1,21
2,2026-03-09 00:18:13,60,67,36.8,74,-417.0,0,0.000000,0.000000,1,21
3,2026-03-09 00:18:17,65,67,36.4,73,-436.0,0,0.394765,0.768830,1,21
4,2026-03-09 00:18:21,52,67,36.4,73,-320.0,0,0.401153,0.540071,1,21


## Parse and Sort Timestamps

Convert the `timestamp` column from string to `datetime64` so pandas can use it
for time-based operations. Sorting ensures the resampler sees strictly increasing time.

In [3]:
# Parse timestamps from string to datetime64
df['timestamp'] = pd.to_datetime(df['timestamp'])

# Sort chronologically — required before setting as a DatetimeIndex
df = df.sort_values('timestamp')


## Set Timestamp as Index

Pandas `resample()` requires a `DatetimeIndex`. Promoting `timestamp` to the index
enables time-frequency operations on the DataFrame.

In [4]:
# Promote timestamp to the index to enable resample()
df = df.set_index('timestamp')


## Resample to 4-Second Grid & Interpolate

Three steps are applied in sequence to produce a gap-free, uniformly spaced dataset:

1. **`resample('4s').mean()`** — bin all raw readings that fall within each 4-second
   window and replace them with their mean. Empty bins (gaps in collection) become `NaN`.
2. **`ffill()`** — forward-fill any leading or isolated `NaN` values using the last
   known observation. Handles gaps at the start of the series where interpolation
   cannot look backwards.
3. **`interpolate()`** — fill any remaining `NaN` values using linear interpolation
   between neighbouring valid observations, producing smooth transitions across gaps.

`reset_index()` restores `timestamp` as a regular column for CSV export.

In [5]:
# Step 1: Aggregate readings into 4-second bins; gaps become NaN
df_4s = df.resample('4s').mean()

# Step 2: Forward-fill to handle leading NaNs (interpolate cannot fill these)
df_4s = df_4s.ffill()

# Step 3: Linear interpolation for any remaining interior NaNs
df_4s = df_4s.interpolate()

# Restore timestamp as a column for downstream CSV compatibility
df_4s = df_4s.reset_index()


## Preview Resampled Dataset

Verify the output: timestamps should now fall on exact 4-second boundaries
(e.g. `00:18:04`, `00:18:08`, …) and no `NaN` values should remain.

In [6]:
df_4s.head()


,timestamp,cpu_usage,memory_usage,temperature,battery_level,battery_drain,battery_charging,upload_speed_kbps,download_speed_kbps,screen_on,brightness
0,2026-03-09 00:18:04,60.0,63.0,36.8,74.0,-223.0,0.0,0.000000,0.000000,1.0,21.0
1,2026-03-09 00:18:08,49.0,64.0,36.8,74.0,-700.0,0.0,4.524958,1.739434,1.0,21.0
2,2026-03-09 00:18:12,60.0,67.0,36.8,74.0,-417.0,0.0,0.000000,0.000000,1.0,21.0
3,2026-03-09 00:18:16,65.0,67.0,36.4,73.0,-436.0,0.0,0.394765,0.768830,1.0,21.0
4,2026-03-09 00:18:20,52.0,67.0,36.4,73.0,-320.0,0.0,0.401153,0.540071,1.0,21.0


## Save Output

Write the regularised dataset to `dataset_time_series.csv`. The row index is
excluded (`index=False`) since `timestamp` is already present as a named column.
This file is the input consumed by the preprocessing pipeline (`preprocessing.py`).

In [7]:
# Save the uniformly resampled dataset — used as input by the clustering pipeline
df_4s.to_csv('../data/dataset/dataset_time_series.csv', index=False)
